# Amazon Bedrock AgentCore 런타임과 아이덴티티 격리를 활용한 AgentCore 메모리 에이전트

## 개요

이 튜토리얼은 AgentCore 런타임과 AgentCore 메모리를 사용하여 사용자 격리 기능이 포함된 첫 번째 메모리 지원 에이전트를 만드는 방법을 보여줍니다. 세션 내에서 이전 상호작용을 기억하는 간단한 "Hello World" 대화형 에이전트를 구축하여, 사용자와의 상호작용 간에 더 자연스럽고 맥락에 맞는 대화를 가능하게 합니다.

메모리는 효과적인 대화형 에이전트를 만드는 데 핵심적인 구성 요소입니다. 맥락을 유지하고, 사용자 선호도를 기억하며, 시간이 지나도 일관된 응답을 제공할 수 있게 합니다. 메모리가 없으면 에이전트는 매 상호작용마다 처음부터 시작해야 하므로 단절된 사용자 경험을 초래합니다.

이 구현은 인증된 사용자 자격 증명을 기반으로 메모리를 자동으로 분할하는 사용자 아이덴티티 전파 기능이 포함된 Amazon Bedrock AgentCore 메모리를 활용하여, 각 개별 사용자를 위한 안전하고 격리된 메모리 공간을 생성합니다.

### 튜토리얼 세부 정보


| 항목                | 세부 내용                                                        |
|:--------------------|:-----------------------------------------------------------------|
| 튜토리얼 유형       | Hello World / 입문                                               |
| 에이전트 유형       | 단일 대화형 에이전트                                             |
| 에이전트 프레임워크 | Strands 에이전트                                                 |
| LLM 모델            | Anthropic Claude Haiku 3.5                                      |
| 주요 기능           | AgentCore 런타임, 메모리 통합                                    |
| 난이도              | 중급                                                             |
| 사용 SDK            | boto3, bedrock-agentcore, bedrock-agentcore-starter-toolkit      |

### 학습 내용

이 튜토리얼에서 배울 내용:
1. AgentCore 메모리를 사용하여 에이전트의 메모리 리소스를 생성하는 방법
2. 대화 기록을 저장하고 검색하기 위한 메모리 훅 구현 방법
3. 확장 가능한 프로덕션 사용을 위해 에이전트를 AgentCore 런타임에 배포하는 방법
4. 세션 관리로 에이전트를 테스트하고 메모리 지속성을 확인하는 방법
5. 사용자 아이덴티티를 처리하고 다른 사용자 간 메모리 격리를 보장하는 방법


### 아키텍처

이 Hello World 예제는 메모리 통합이 포함된 AgentCore 런타임에 배포된 간단한 대화형 에이전트를 보여줍니다:

<div style="text-align:left">
    <img src="runtime-memory-identity.png" width="90%"/>
</div>

## 0. 사전 요구사항

이 튜토리얼을 실행하려면 다음이 필요합니다:
* Python 3.10 이상
* Bedrock, ECR, IAM 및 Cognito에 대한 적절한 권한이 구성된 AWS 자격 증명
* Amazon Bedrock 모델 접근 (Claude 3.5 Haiku)
* Amazon Bedrock AgentCore SDK 및 종속성

먼저 필요한 라이브러리를 설치하겠습니다.

In [ ]:
!pip install -qUr requirements.txt

### 환경 설정

필요한 라이브러리를 가져오고 환경을 구성하겠습니다. 다음을 사용합니다:
- AWS 서비스 상호작용을 위한 `boto3`
- 에이전트 메모리 관리를 위한 `bedrock_agentcore.memory`
- 인증 설정을 위한 다양한 유틸리티 함수

In [ ]:
# 가져오기
import os
import jwt
import time
import boto3
import uuid
import logging
from bedrock_agentcore.memory import MemoryClient
from utils import setup_cognito_user_pool, reauthenticate_users

# 구성
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)
logger = logging.getLogger("runtime-memory-agent")
REGION = os.getenv('AWS_REGION', 'us-west-2')
memory_client = MemoryClient(region_name=REGION)

## 1. Amazon Cognito 사용자 풀 생성

이 섹션에서는 Amazon Cognito 사용자 풀과 사용자를 생성합니다. Cognito는 에이전트에 대한 사용자 인증 및 아이덴티티 관리를 제공하여, 각 사용자의 대화 기록이 해당 사용자만 접근할 수 있도록 보장합니다.

`setup_cognito_user_pool` 함수는 다음을 수행합니다:
1. Cognito 사용자 풀이 없으면 생성
2. 인증을 위한 앱 클라이언트 설정
3. 임시 비밀번호로 2명의 테스트 사용자 생성
4. 테스트를 위한 접근 토큰 생성

In [ ]:
print("Amazon Cognito 사용자 풀 및 사용자 설정 중...")
cognito_config = setup_cognito_user_pool(region=REGION)
print("Cognito 설정 완료")

## 2. 메모리 리소스 생성

이 섹션에서는 에이전트의 대화 기록을 저장할 메모리 리소스를 생성합니다. 메모리를 통해 에이전트는 과거 상호작용을 회상하고, 맥락을 유지하며, 시간이 지남에 따라 더 일관된 응답을 제공할 수 있습니다.

이 예제에서는 추가 장기 전략 없이 간단한 단기 메모리 리소스를 생성합니다. 메모리는 모든 대화 메시지를 저장하여, AgentCore 런타임에서 세션이 종료된 후에도 이전 상호작용을 기억할 수 있도록 도와줍니다.

In [ ]:
from botocore.exceptions import ClientError

# 이 리소스의 고유 식별자 생성
unique_id = str(uuid.uuid4())[:8]
memory_name = f"RuntimeIdentityMemoryAgent_{unique_id}"

try:
    # 전략 없이 메모리 리소스 생성 (단기 메모리만)
    memory = memory_client.create_memory_and_wait(
        name=memory_name,
        strategies=[],  # 단기 메모리를 위한 전략 없음
        description="Short-term memory for AgentCore Runtime agent authenticated with AgentCore Identity",
        event_expiry_days=7, # 단기 메모리 보존 기간
    )
    memory_id = memory['id']
    logger.info(f"메모리 생성 완료: {memory_id}")
except ClientError as e:
    logger.info(f"오류: {e}")
    if e.response['Error']['Code'] == 'ValidationException' and "already exists" in str(e):
        # 메모리가 이미 존재하면 해당 ID를 가져옴
        memories = memory_client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(memory_name)), None)
        logger.info(f"메모리가 이미 존재합니다. 기존 메모리 ID 사용: {memory_id}")
except Exception as e:
    # 메모리 생성 중 오류 표시
    logger.error(f"오류: {e}")
    import traceback
    traceback.print_exc()
    # 오류 시 정리 - 부분적으로 생성된 메모리 삭제
    if 'memory_id' in locals() and memory_id:
        try:
            memory_client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"메모리 정리 완료: {memory_id}")
        except Exception as cleanup_error:
            logger.error(f"메모리 정리 실패: {cleanup_error}")

## 3. 메모리 지원 에이전트 생성

이 섹션에서는 메모리 통합을 위한 커스텀 훅이 포함된 Strands 에이전트 프레임워크를 사용하여 메모리 지원 에이전트를 구축합니다. 이 에이전트는 AgentCore 메모리에서 메시지를 저장하고 검색하여 대화 맥락을 유지합니다.

> **메모리가 중요한 이유**: AgentCore 런타임의 세션은 일정 시간 후 만료되어 대화 맥락이 삭제됩니다. 대화를 메모리에 저장하면 세션 간에도 이전 정보가 유지되어, 긴 중단 후에도 사용자에게 원활한 경험을 제공합니다.

### 에이전트 기능

에이전트는 다음을 수행합니다:
1. 각 사용자 및 어시스턴트 메시지를 메모리에 자동 저장
2. 기존 세션 계속 시 과거 대화 기록 검색
3. 동일 사용자와의 여러 상호작용에 걸쳐 맥락 유지
4. 사용자 아이덴티티 확인을 통한 다른 사용자 간 대화 격리

### 구현의 핵심 구성 요소

#### 1. 메모리 훅 프로바이더
커스텀 훅 프로바이더가 구현하는 내용:
- `on_agent_initialized`: 에이전트 시작 시 트리거되어 AgentCore 메모리에서 대화 기록 검색
- `on_message_added`: 새 메시지가 대화에 추가될 때 트리거되어 AgentCore 메모리에 저장

#### 2. 에이전트 초기화
`initialize_agent` 함수:
- 올바른 리전으로 메모리 훅 구성
- 적절한 상태 변수(memory_id, actor_id, session_id)로 에이전트 설정
- LLM의 시스템 프롬프트 구성

#### 3. 사용자 검증
`get_user_sub` 함수:
- JWKS에 대해 Cognito 접근 토큰을 검증하고 사용자의 sub(고유 ID)를 반환

#### 4. 진입점 핸들러
runtime_memory_agent 함수:
- 입력 페이로드 파싱 및 사용자 메시지 추출
- Cognito의 JWT 토큰을 사용한 사용자 아이덴티티 검증
- 에이전트 초기화 및 세션 추적 관리
- 적절한 맥락으로 에이전트 호출 처리
- 런타임 환경에 형식화된 응답 반환

에이전트 파일을 생성하겠습니다:

In [ ]:
%%writefile runtime_identity_memory_agent.py
import os
import jwt
import json
import logging
from strands import Agent
from jwt import PyJWKClient
from typing import Dict, Any
from strands.models import BedrockModel
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands.hooks import AgentInitializedEvent, HookProvider, HookRegistry, MessageAddedEvent

# 상세 로깅 구성
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)
logger = logging.getLogger("runtime-memory-agent")

# AgentCore 앱 초기화
app = BedrockAgentCoreApp()

MODEL_ID = os.getenv('MODEL_ID')
MEMORY_ID = os.getenv('MEMORY_ID')
COGNITO_USER_POOL = os.getenv('COGNITO_USER_POOL')
REGION = os.getenv('AWS_REGION')

# 전역 에이전트 인스턴스 - 첫 번째 요청 시 초기화됨
agent = None

class MemoryHookProvider(HookProvider):
    """Bedrock 메모리와 통합하기 위한 커스텀 훅 프로바이더"""
    
    def __init__(self, region_name):
        logger.info(f"MemoryHookProvider를 리전 {region_name}으로 초기화 중")
        self.memory_client = MemoryClient(region_name=region_name)
    
    def on_agent_initialized(self, event: AgentInitializedEvent):
        """에이전트 시작 시 최근 대화 기록 로드"""
        logger.info("에이전트 초기화 훅 트리거됨")
        
        memory_id = event.agent.state.get("memory_id")
        actor_id = event.agent.state.get("actor_id")
        session_id = event.agent.state.get("session_id")
        
        logger.info(f"상태 값 - memory_id: {memory_id}, actor_id: {actor_id}, session_id: {session_id}")
        
        missing_values = []
        if not memory_id:
            missing_values.append("memory_id")
        if not actor_id:
            missing_values.append("actor_id")
        if not session_id:
            missing_values.append("session_id")
            
        if missing_values:
            logger.warning(f"필수 값 누락: {', '.join(missing_values)}")
            return
        
        try:
            # 먼저 세션이 존재하는지 이벤트 1개로 제한하여 확인
            logger.info(f"세션 {session_id} 존재 여부 확인 중...")
            session_exists = False
            try:
                events = self.memory_client.list_events(
                    memory_id=memory_id,
                    actor_id=actor_id,
                    session_id=session_id,
                    max_results=1
                )
                session_exists = len(events) > 0
                logger.info(f"세션 존재 여부: {session_exists} ({len(events)}개 이벤트 발견)")
            except Exception as e:
                logger.warning(f"세션 존재 확인 중 오류: {e}")
                # 이벤트가 없다고 가정하고 계속 진행
                session_exists = False
            
            # 세션이 존재하지 않으면 대화 기록 로드 불필요
            if not session_exists:
                logger.info(f"세션 {session_id}에 대한 기존 대화를 찾을 수 없음")
                return
            
            # 세션이 존재하면 대화 기록 로드
            logger.info(f"기존 세션 {session_id}의 대화 기록 로드 중")
            recent_turns = self.memory_client.get_last_k_turns(
                memory_id=memory_id,
                actor_id=actor_id,
                session_id=session_id,
                k=5
            )
            
            if recent_turns:
                logger.info(f"✅ 메모리에서 {len(recent_turns)}개의 대화 턴 로드 완료")
                context_messages = []
                for turn in recent_turns:
                    for message in turn:
                        role = message['role']
                        content = message['content']['text']
                        context_messages.append(f"{role}: {content}")
                
                context = "\n".join(context_messages)
                event.agent.system_prompt += f"\n\nRecent conversation:\n{context}"
                logger.info("✅ 시스템 프롬프트에 대화 컨텍스트 추가 완료")
            else:
                logger.info("이 세션에 대한 최근 턴을 찾을 수 없음")
                
        except Exception as e:
            logger.error(f"❌ 메모리 로드 오류: {e}", exc_info=True)
    
    def on_message_added(self, event: MessageAddedEvent):
        """메시지를 메모리에 저장"""
        logger.info("메시지 추가 훅 트리거됨")
        
        memory_id = event.agent.state.get("memory_id")
        actor_id = event.agent.state.get("actor_id")
        session_id = event.agent.state.get("session_id")
        
        logger.info(f"상태 값 - memory_id: {memory_id}, actor_id: {actor_id}, session_id: {session_id}")
        
        missing_values = []
        if not memory_id:
            missing_values.append("memory_id")
        if not actor_id:
            missing_values.append("actor_id")
        if not session_id:
            missing_values.append("session_id")
            
        if missing_values:
            logger.warning(f"❌ 메시지 저장 불가 - 누락된 값: {', '.join(missing_values)}")
            return
            
        messages = event.agent.messages
        try:
            last_message = messages[-1]
            message_content = str(last_message.get("content", ""))
            message_role = last_message["role"]
            
            logger.info(f"{message_role} 메시지를 메모리에 저장 중: {message_content[:30]}...")
            
            self.memory_client.create_event(
                memory_id=memory_id,
                actor_id=actor_id,
                session_id=session_id,
                messages=[(message_content, message_role)]
            )
            logger.info("✅ 메시지가 메모리에 성공적으로 저장됨")
        except Exception as e:
            logger.error(f"❌ 메모리 저장 오류: {e}", exc_info=True)
    
    def register_hooks(self, registry: HookRegistry):
        logger.info("메모리 훅 등록 중")
        registry.add_callback(MessageAddedEvent, self.on_message_added)
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)

def initialize_agent(actor_id, session_id):
    """첫 번째 사용을 위한 에이전트 초기화"""
    global agent
    
    logger.info(f"actor_id={actor_id}, session_id={session_id}로 에이전트 초기화 중")
    
    # 모델 및 메모리 훅 생성
    logger.info(f"모델 ID: {MODEL_ID}로 모델 생성 중")
    model = BedrockModel(model_id=MODEL_ID)
    logger.info(f"리전: {REGION}으로 메모리 훅 생성 중")
    memory_hook = MemoryHookProvider(region_name=REGION)
    
    # 적절한 초기 상태로 에이전트 생성
    logger.info("메모리 훅으로 에이전트 생성 중")
    agent = Agent(
        model=model,
        hooks=[memory_hook],
        system_prompt="You're a helpful, memory-enabled agent deployed on AgentCore Runtime. You can remember previous interactions within the same session. Be friendly and concise in your responses.",
        state={
            "memory_id": MEMORY_ID,
            "actor_id": actor_id,
            "session_id": session_id
        }
    )
    logger.info(f"✅ 에이전트 초기화 완료, 상태: {agent.state.get()}")

def get_user_sub(access_token: str, region: str, user_pool_id: str) -> str:
    """
    JWKS에 대해 Cognito 접근 토큰을 검증하고 사용자의 sub(고유 ID)를 반환합니다.

    :param access_token: JWT 접근 토큰 문자열
    :param region: Cognito 사용자 풀의 AWS 리전
    :param user_pool_id: Cognito 사용자 풀 ID
    :return: 토큰이 유효한 경우 사용자의 'sub' 클레임
    :raises jwt.InvalidTokenError: 검증 실패 시
    """
    access_token = access_token[7:]
    jwks_url = f"https://cognito-idp.{region}.amazonaws.com/{user_pool_id}/.well-known/jwks.json"
    jwks_client = PyJWKClient(jwks_url)
    signing_key = jwks_client.get_signing_key_from_jwt(access_token)

    decoded = jwt.decode(
        access_token,
        signing_key.key,
        algorithms=["RS256"],
        issuer=f"https://cognito-idp.{region}.amazonaws.com/{user_pool_id}",
        options={"require": ["exp", "iat", "iss", "token_use"]}
    )

    if decoded.get("token_use") != "access":
        raise jwt.InvalidTokenError("Token is not an access token")

    return decoded["sub"]

@app.entrypoint
def runtime_memory_agent(payload, context):
    """
    Main entry point for the memory-enabled agent
    
    Args:
        payload: The input payload containing user data
        context: The runtime context object containing session information
    """
    global agent
    
    # 페이로드와 컨텍스트 정보 모두 로깅
    logger.info(f"수신된 페이로드: {payload}")
    logger.info(f"컨텍스트: {context}")
    logger.info(f"컨텍스트 인증: {context.request_headers.get('Authorization')}")
    logger.info(f"사용자 Sub: {get_user_sub(context.request_headers.get('Authorization'), REGION, COGNITO_USER_POOL)}")
    logger.info(f"컨텍스트 session_id: {context.session_id}")
    
    # 필수 값 추출 및 검증
    user_input = payload.get("prompt")
    actor_id = get_user_sub(context.request_headers.get('Authorization'), REGION, COGNITO_USER_POOL)
    session_id = context.session_id  # 컨텍스트에서 session_id 가져오기
    
    # 필수 필드 검증
    if user_input is None:
        error_msg = "❌ 오류: 페이로드에 'prompt' 필드가 없습니다"
        logger.error(error_msg)
        return error_msg
    
    # 첫 번째 요청 시 에이전트 초기화
    if agent is None:
        logger.info("첫 번째 요청 - 에이전트 초기화 중")
        initialize_agent(actor_id, session_id)
    else:
        logger.info("기존 에이전트 인스턴스 사용")
        # 세션 ID가 변경된 경우 업데이트
        if agent.state.get("session_id") != session_id:
            logger.info(f"세션 ID를 {session_id}로 업데이트 중")
            agent.state.set("session_id", session_id)
        if agent.state.get("actor_id") != actor_id:
            logger.info(f"액터 ID를 {actor_id}로 업데이트 중")
            agent.state.set("actor_id", actor_id)
    
    # 사용자 입력으로 에이전트 호출
    logger.info(f"입력으로 에이전트 호출 중: {user_input}")
    response = agent(user_input)
    response_text = response.message['content'][0]['text']
    logger.info(f"✅ 에이전트 응답: {response_text[:50]}...")
    
    return response_text

if __name__ == "__main__":
    logger.info("AgentCore 애플리케이션 시작 중")
    app.run()

## 4. AgentCore 런타임에 배포

이 섹션에서는 확장성과 간소화된 운영을 제공하는 관리형 에이전트 런타임 환경인 Amazon Bedrock AgentCore 런타임에 에이전트를 배포합니다. AgentCore 런타임은 인프라 복잡성을 처리하여, 배포 관리보다 에이전트 로직에 집중할 수 있게 합니다.

수동 서버 설정 및 관리가 필요한 기존 배포 방법과 달리, AgentCore 런타임은 코드를 컨테이너에 자동으로 패키징하고, AWS 인프라에 배포하며, 호출을 위한 보안 HTTPS 엔드포인트를 제공합니다. 이 접근 방식은 에이전트가 수요에 따라 확장하고 프로덕션 환경에서 안정적으로 운영될 수 있도록 보장합니다.

### 내부 동작

AgentCore 런타임에 배포하면 다음이 자동으로 수행됩니다:
1. 코드가 Docker 컨테이너 이미지로 패키징
2. 컨테이너 이미지가 Amazon ECR(Elastic Container Registry)에 푸시
3. 에이전트를 실행하기 위한 AWS Lambda 함수 또는 컨테이너 서비스 프로비저닝
4. 보안 접근을 위한 API Gateway 엔드포인트 생성
5. 보안 운영을 위한 IAM 역할 및 권한 구성

### 알아야 할 사항

- **AgentCore 런타임**은 에이전트를 Docker 컨테이너로 패키징하고 관리형 AWS 인프라에 배포
- **환경 변수**가 에이전트를 구성:
  - `MEMORY_ID`: 앞서 생성한 메모리 리소스
  - `MODEL_ID`: Claude 3.5 Haiku 모델 ID
  - `AWS_REGION`: 배포를 위한 AWS 리전
  - `COGNITO_USER_POOL`: 인증을 위한 Cognito 사용자 풀

> **팁**: AgentCore 스타터 툴킷이 IAM 역할, ECR 저장소, 컨테이너 빌드를 포함한 모든 복잡한 배포 단계를 처리합니다.

### 배포 구성

배포 구성을 설정하겠습니다:

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
import time

agentcore_runtime = Runtime()
agent_name = f"runtime_memory_agent_{unique_id}"

response = agentcore_runtime.configure(
    entrypoint="runtime_identity_memory_agent.py", 
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=REGION,
    agent_name=agent_name,
    non_interactive=True, 
    memory_mode="NO_MEMORY",
    request_header_configuration = {"requestHeaderAllowlist": ["Authorization"]},
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": cognito_config.get("discovery_url"),
            "allowedClients": [cognito_config.get("client_id")]
        }
    }
)
response

### 에이전트 시작

이제 AgentCore 런타임에 에이전트를 시작합니다. 이 단계에서는 구성된 에이전트를 AgentCore의 관리형 인프라에 배포합니다. 이 과정에서 에이전트에 필요한 필수 환경 변수를 전달합니다: 앞서 생성한 메모리 ID, 사용할 모델 ID, AWS 리전, 인증을 위한 Cognito 사용자 풀 ID입니다.

배포 후 에이전트는 사용자 메시지로 호출할 수 있는 보안 엔드포인트를 통해 접근 가능합니다. 엔드포인트는 Cognito 인증으로 보호되어 인가된 사용자만 에이전트에 접근할 수 있습니다.

In [ ]:
launch_result = agentcore_runtime.launch(
    env_vars={
        "MEMORY_ID": memory_id,
        "MODEL_ID": "global.anthropic.claude-haiku-4-5-20251001-v1:0",
        "AWS_REGION": REGION,
        "COGNITO_USER_POOL": cognito_config["pool_id"]
    }
)

### 배포 상태 확인

에이전트의 배포 상태를 확인하겠습니다. AgentCore 런타임이 컨테이너를 빌드하고, 필요한 리소스를 프로비저닝하고, AWS 인프라에 에이전트를 배포하는 데 몇 분이 걸릴 수 있습니다. 배포가 완료될 때까지 10초마다 상태를 폴링합니다.

In [ ]:
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']

while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(f"현재 상태: {status}")

if status == 'READY':
    print("에이전트 배포 성공!")
else:
    print(f"배포 종료 상태: {status}")

## 5. 에이전트 테스트

에이전트가 배포되었으므로, 메시지를 보내고 이전 상호작용을 기억하는지 확인하여 테스트하겠습니다. 또한 다른 사용자가 격리된 메모리 컨텍스트를 갖는지 테스트하여, 한 사용자의 대화가 다른 사용자에게 보이지 않도록 합니다.

**세션 관리에 대한 중요 참고사항**

- **세션 관리**: AgentCore 런타임은 세션 ID가 제공되지 않으면 자동으로 생성하지만, 애플리케이션에서 세션 ID를 명시적으로 관리하는 것이 권장됩니다. 이를 통해 다음을 더 잘 제어할 수 있습니다:
  - 세션 타임아웃 후 대화 계속
  - 적절한 시점에 새 세션 생성 (예: 사용자가 새 대화를 시작할 때)
  - 동일 사용자와의 여러 병렬 대화 처리
  - 애플리케이션 요구에 따른 세션 만료 정책 구현

- **메모리 지속성**: AgentCore 런타임에서 세션이 만료되더라도, 동일 사용자로 새 세션이 시작될 때 에이전트는 AgentCore 메모리에서 이전 대화를 검색할 수 있습니다.

먼저 JWT 토큰 검증을 위한 도우미 함수를 정의하겠습니다:

In [ ]:
def test_user_memory_isolation():
    """
    AgentCore에서 각 사용자의 메모리가 격리되어 있는지 테스트합니다.
    
    이 테스트는 다음을 확인합니다:
    1. 각 사용자의 대화가 별도로 저장됨
    2. 에이전트가 각 사용자와의 이전 상호작용을 기억함
    3. 사용자 데이터가 다른 사용자 간에 공유되지 않음
    """
    print("\n" + "=" * 50)
    print("사용자 메모리 격리 테스트")
    print("=" * 50)
    
    # testuser1과 testuser2의 세션 ID 생성
    testuser1_session_id = f"agent-session-testuser1-{int(time.time())}"
    testuser2_session_id = f"agent-session-testuser2-{int(time.time())}"
    
    testuser1_token = cognito_config["bearer_tokens"]["testuser1"]
    testuser2_token = cognito_config["bearer_tokens"]["testuser2"]
    
    # 단계 1: testuser1이 좋아하는 색상을 공유
    print("\n" + "-" * 50)
    print("단계 1: 첫 번째 사용자가 개인 정보를 공유합니다")
    print("-" * 50)
    # "내가 좋아하는 색상은 보라색입니다."
    print("testuser1: \"My favorite color is purple.\"")
    
    response1 = agentcore_runtime.invoke(
        {"prompt": "My favorite color is purple."},
        session_id=testuser1_session_id,
        bearer_token=testuser1_token
    )
    print(f"에이전트: \"{response1['response']}\"")
    
    # 단계 2: testuser2가 좋아하는 음식을 공유
    print("\n" + "-" * 50)
    print("단계 2: 두 번째 사용자가 다른 정보를 공유합니다")
    print("-" * 50)
    # "내가 좋아하는 음식은 피자입니다."
    print("testuser2: \"My favorite food is pizza.\"")
    
    response2 = agentcore_runtime.invoke(
        {"prompt": "My favorite food is pizza."},
        session_id=testuser2_session_id,
        bearer_token=testuser2_token
    )
    print(f"에이전트: \"{response2['response']}\"")
    
    # 단계 3: testuser1이 색상에 대해 물어봄
    print("\n" + "-" * 50)
    print("단계 3: 첫 번째 사용자가 에이전트의 메모리를 테스트합니다")
    print("-" * 50)
    # "내가 좋아하는 색상이 뭐라고 했죠?"
    print("testuser1: \"What did I say my favorite color was?\"")
    
    response3 = agentcore_runtime.invoke(
        {"prompt": "What did I say my favorite color was?"},
        session_id=testuser1_session_id,
        bearer_token=testuser1_token
    )
    print(f"에이전트: \"{response3['response']}\"")
    
    # 단계 4: testuser2가 음식에 대해 물어봄
    print("\n" + "-" * 50)
    print("단계 4: 두 번째 사용자가 에이전트의 메모리를 테스트합니다")
    print("-" * 50)
    # "내가 좋아하는 음식이 뭔가요?"
    print("testuser2: \"What's my favorite food?\"")
    
    response4 = agentcore_runtime.invoke(
        {"prompt": "What's my favorite food?"},
        session_id=testuser2_session_id,
        bearer_token=testuser2_token
    )
    print(f"에이전트: \"{response4['response']}\"")
    
    # 단계 5: testuser1이 음식에 대해 물어봄 (알 수 없어야 함)
    print("\n" + "-" * 50)
    print("단계 5: 메모리 격리 테스트 (첫 번째 사용자)")
    print("-" * 50)
    # "내가 좋아하는 음식이 뭔가요?"
    print("testuser1: \"What's my favorite food?\"")
    
    response5 = agentcore_runtime.invoke(
        {"prompt": "What's my favorite food?"}, 
        session_id=testuser1_session_id,
        bearer_token=testuser1_token
    )
    print(f"에이전트: \"{response5['response']}\"")
    
    # 단계 6: testuser2가 색상에 대해 물어봄 (알 수 없어야 함)
    print("\n" + "-" * 50)
    print("단계 6: 메모리 격리 테스트 (두 번째 사용자)")
    print("-" * 50)
    # "내가 좋아하는 색상이 뭔가요?"
    print("testuser2: \"What's my favorite color?\"")
    
    response6 = agentcore_runtime.invoke(
        {"prompt": "What's my favorite color?"},
        session_id=testuser2_session_id,
        bearer_token=testuser2_token
    )
    print(f"에이전트: \"{response6['response']}\"")
    

In [ ]:
test_user_memory_isolation()

## 핵심 개념

이 튜토리얼에서 AgentCore로 메모리 지원 에이전트를 구축하기 위한 몇 가지 중요한 개념을 배웠습니다:

1. **메모리 통합**: Amazon Bedrock 메모리를 사용하여 세션 간 대화 기록을 저장하는 방법으로, 세션이 만료되더라도 에이전트가 시간이 지남에 따라 맥락을 유지할 수 있게 합니다.

2. **세션 관리**: 세션 ID를 사용하여 대화를 구성하고 사용자가 돌아올 때 관련 기록을 검색하여 원활한 경험을 만드는 방법입니다.

3. **AgentCore 배포**: 확장, 보안 및 인프라 관리를 자동으로 처리하는 프로덕션 런타임 환경에 에이전트를 배포하는 방법입니다.

4. **메모리 훅**: 메모리 서비스와 통합하는 커스텀 훅을 구현하여, 에이전트 라이프사이클의 특정 시점에서 대화 기록을 저장하고 검색하는 방법입니다.

5. **사용자 아이덴티티 및 프라이버시**: 인증을 사용하여 각 사용자의 대화 기록이 비공개이며 다른 사용자로부터 격리되도록 보장하는 방법입니다.

이러한 개념은 지속적인 메모리와 정교한 대화 관리 기능을 갖춘 더 복잡한 에이전트를 구축하기 위한 기반을 제공합니다.

## 정리 (선택 사항)

이 튜토리얼에서 생성한 리소스가 더 이상 필요하지 않다면, 불필요한 AWS 비용을 방지하기 위해 정리할 수 있습니다. 여기에는 다음이 포함됩니다:

1. AgentCore 런타임 에이전트
2. 에이전트 컨테이너 이미지가 포함된 ECR 저장소
3. 대화 기록을 저장하는 메모리 리소스

먼저 리소스를 확인하겠습니다:

In [ ]:
# 리소스 식별자 가져오기
if 'launch_result' in locals():
    print(f"에이전트 ID: {launch_result.agent_id}")
    print(f"ECR 저장소: {launch_result.ecr_uri.split('/')[1]}")
else:
    print("시작 결과를 사용할 수 없습니다")

In [ ]:
# 모든 리소스를 삭제하려면 이 셀을 실행하세요

# 1. AgentCore 런타임 삭제
if 'launch_result' in locals() and hasattr(launch_result, 'agent_id'):
    try:
        agentcore_control_client = boto3.client(
            'bedrock-agentcore-control',
            region_name=REGION
        )
        
        runtime_delete_response = agentcore_control_client.delete_agent_runtime(
            agentRuntimeId=launch_result.agent_id,
        )
        print(f"AgentCore 런타임 삭제 완료: {launch_result.agent_id}")
    except Exception as e:
        print(f"AgentCore 런타임 삭제 오류: {e}")
else:
    print("삭제할 AgentCore 런타임이 없습니다")

# 2. ECR 저장소 삭제
if 'launch_result' in locals() and hasattr(launch_result, 'ecr_uri'):
    try:
        ecr_client = boto3.client(
            'ecr',
            region_name=REGION
        )
        
        repository_name = launch_result.ecr_uri.split('/')[1]
        response = ecr_client.delete_repository(
            repositoryName=repository_name,
            force=True  # 이미지가 있어도 강제 삭제
        )
        print(f"ECR 저장소 삭제 완료: {repository_name}")
    except Exception as e:
        print(f"ECR 저장소 삭제 오류: {e}")
else:
    print("삭제할 ECR 저장소가 없습니다")

# 3. 메모리 리소스 삭제
if 'memory_id' in locals() and memory_id:
    try:
        memory_client = MemoryClient(region_name=REGION)
        memory_client.delete_memory_and_wait(memory_id=memory_id)
        print(f"메모리 리소스 삭제 완료: {memory_id}")
    except Exception as e:
        print(f"메모리 리소스 삭제 오류: {e}")
else:
    print("삭제할 메모리 리소스가 없습니다")

# 4. Cognito 사용자 풀 및 관련 리소스 삭제
if 'cognito_config' in locals() and cognito_config and 'pool_id' in cognito_config:
    try:
        cognito_client = boto3.client('cognito-idp', region_name=REGION)
        
        # 사용자 풀 ID 가져오기
        pool_id = cognito_config['pool_id']
        
        # 모든 사용자 풀 클라이언트 나열 및 삭제
        clients_response = cognito_client.list_user_pool_clients(
            UserPoolId=pool_id,
            MaxResults=60
        )
        
        for client in clients_response.get('UserPoolClients', []):
            client_id = client['ClientId']
            cognito_client.delete_user_pool_client(
                UserPoolId=pool_id,
                ClientId=client_id
            )
            print(f"사용자 풀 클라이언트 삭제 완료: {client_id}")
        
        # 사용자 풀 자체 삭제
        cognito_client.delete_user_pool(
            UserPoolId=pool_id
        )
        print(f"Cognito 사용자 풀 삭제 완료: {pool_id}")
        
    except Exception as e:
        print(f"Cognito 리소스 삭제 오류: {e}")
else:
    print("삭제할 Cognito 리소스가 없습니다")

print("\n정리 완료")

## 축하합니다!

Amazon Bedrock AgentCore 런타임, AgentCore 아이덴티티 및 AgentCore 메모리를 사용한 첫 번째 메모리 지원 에이전트를 성공적으로 구축하고 배포했습니다! 이 에이전트는 몇 가지 중요한 기능을 보여줍니다:

1. **메모리 지속성**: 에이전트가 이전 대화를 기억할 수 있습니다.
2. **사용자 아이덴티티**: 에이전트가 다른 사용자에 대해 별도의 대화 기록을 유지합니다.
3. **관리형 인프라**: 에이전트가 AWS 관리형 인프라에서 실행되며 필요에 따라 자동으로 확장됩니다.

### 다음 단계

기본 사항을 이해했으므로 다음과 같은 방법으로 에이전트를 향상시킬 수 있습니다:

1. **도구 추가**: 계산기, 데이터베이스 커넥터 또는 API 호출과 같은 도구로 에이전트를 향상시켜 대화 외의 작업을 수행할 수 있게 합니다.
2. **메모리 개선**: 장기 메모리를 사용한 더 정교한 메모리 전략 구현
3. **UI 구축**: React, Flutter 또는 Swift와 같은 프레임워크를 사용하여 에이전트를 위한 웹 또는 모바일 인터페이스 생성
4. **비즈니스 로직 추가**: CRM, 지식 기반 또는 내부 도구와 같은 비즈니스 시스템과 에이전트 통합